# OpenMix: Computational Formulation Science

**RDKit is for molecules. OpenMix is for mixtures.**

OpenMix observes formulations through molecular physics, validates ingredient interactions, and evaluates from multiple computational perspectives -- surfacing where physics, chemistry, and data agree, and where they genuinely disagree.

In this tutorial:
1. Observe a formula through physics (3 lines of code)
2. Catch dangerous interactions
3. Multi-perspective discourse -- the headline feature
4. Drug-excipient compatibility for pharma

No API keys needed. Runs in under 2 minutes.

[![GitHub](https://img.shields.io/badge/GitHub-vijayvkrishnan/openmix-blue)](https://github.com/vijayvkrishnan/openmix)
[![PyPI](https://img.shields.io/pypi/v/openmix)](https://pypi.org/project/openmix/)

In [ ]:
!pip install -q openmix

## 1. Observe a Formulation

The physics observation engine resolves each ingredient to its molecular identity (INCI name -> SMILES -> LogP, MW, charge), then reports what it **sees**, what it **expected**, and where they **disagree**.

In [ ]:
from openmix import Formula, observe

cream = Formula(
    name="Retinol Night Cream",
    ingredients=[
        ("Water", 60.0),
        ("Retinol", 2.0),
        ("Squalane", 15.0),
        ("Cetyl Alcohol", 5.0),
        ("Glycerin", 8.0),
        ("Niacinamide", 5.0),
        ("Ascorbic Acid", 5.0),
    ],
    target_ph=5.5,
    category="skincare",
)

print(observe(cream))

The engine caught: ingredient interactions with confidence scores and literature context, hydrophobic solubility concerns from molecular LogP, and a missing preservative system. Each observation reports what was seen, what was expected, and whether they agree.

## 2. Validate: Catch Dangerous Interactions

258 interaction rules across 6 domains (skincare, pharma, supplements, food, beverages, home care). Hard rules always fire. Soft rules include confidence scores and literature sources.

In [ ]:
from openmix import validate

# A dangerous household cleaner combination
dangerous = Formula(
    ingredients=[
        ("Sodium Hypochlorite", 5.0),
        ("Ammonia", 3.0),
        ("Water", 92.0),
    ],
    category="home_care",
)
print(validate(dangerous))

Three validation modes control sensitivity:

| Mode | Hard Rules | Soft Rules | Use Case |
|------|-----------|-----------|----------|
| `safety` | Error | Warning | Consumer products |
| `formulation` | Error | Info (with mitigations) | Professional formulators |
| `discovery` | Error | Ignored | Research, novel combinations |

In [ ]:
# Same formula, three modes
retinol_aha = Formula(
    ingredients=[
        ("Retinol", 1.0), ("Glycolic Acid", 8.0), ("Water", 91.0),
    ],
    category="skincare",
)

for mode in ["safety", "formulation", "discovery"]:
    report = validate(retinol_aha, mode=mode)
    print(f"{mode}: {report.errors} errors, {report.warnings} warnings, {report.infos} info")

## 3. Discourse: Multi-Perspective Evaluation

This is where OpenMix is different. Instead of a single evaluation, **multiple computational perspectives** assess the same formulation -- physics, chemistry, experimental data, and manufacturing process. The discourse engine classifies their interactions:

- **Agreement**: perspectives concur
- **Correction**: one perspective has stronger evidence and overrides another
- **True disagreement**: comparable evidence, different conclusions -- *worth investigating*
- **Knowledge gap**: nobody has enough information

Evidence hierarchy: empirical data > computational prediction > rule-based > heuristic > LLM reasoning.

In [ ]:
from openmix.discourse import evaluate_discourse
from openmix.protocol import Protocol, Phase, ProcessStep

serum = Formula(
    name="Vitamin C + Retinol Serum",
    ingredients=[
        ("Water", 58.0), ("Ascorbic Acid", 15.0), ("Squalane", 10.0),
        ("Glycerin", 8.0), ("Niacinamide", 4.0), ("Retinol", 1.0),
        ("Cetyl Alcohol", 2.0), ("Phenoxyethanol", 1.0),
        ("Tocopherol", 0.5), ("Xanthan Gum", 0.3),
        ("Disodium EDTA", 0.1), ("Citric Acid", 0.1),
    ],
    target_ph=3.5,
    category="skincare",
)

# Protocol with a deliberate error: retinol in the 75C heat phase
protocol = Protocol(
    phases=[
        Phase("A", "Water Phase", 75.0,
              ["Water", "Glycerin", "Ascorbic Acid", "Niacinamide",
               "Xanthan Gum", "Citric Acid", "Disodium EDTA"]),
        Phase("B", "Oil Phase", 75.0,
              ["Squalane", "Retinol", "Cetyl Alcohol", "Tocopherol"]),
        Phase("C", "Cool-Down", 40.0, ["Phenoxyethanol"]),
    ],
    steps=[
        ProcessStep("heat", "A", {"temp_c": 75, "duration_min": 10}),
        ProcessStep("heat", "B", {"temp_c": 75, "duration_min": 10}),
        ProcessStep("combine", "B", {"into": "A", "mixing_rpm": 500}),
        ProcessStep("cool", "all", {"target_c": 40}),
        ProcessStep("add", "C", {}),
    ],
    equipment=["overhead stirrer"],
)

disc = evaluate_discourse(serum, protocol=protocol)
print(disc)

The discourse engine caught:
- **Chemistry agreements**: Retinol + Ascorbic Acid instability at low pH, Niacinamide + Ascorbic Acid debate
- **Process concerns**: Retinol and Ascorbic Acid degrade above 40C but are assigned to the 75C phase
- **True disagreement**: Physics says the emulsion structure is manageable, but Chemistry says HLB matching is needed and Process says no homogenizer is specified

The true disagreement is the interesting output -- it's where our computational perspectives see the same formula differently, and both have defensible evidence.

## 4. Drug-Excipient Compatibility

OpenMix isn't just for cosmetics. It covers pharmaceutical formulations with 86 drug-excipient interaction rules covering Maillard reactions, chelation, oxidation, and more.

In [ ]:
# A tablet formulation with known incompatibilities
tablet = Formula(
    name="Amlodipine 5mg Tablet",
    ingredients=[
        ("Amlodipine", 2.0),
        ("Lactose", 60.0),
        ("Microcrystalline Cellulose", 25.0),
        ("Croscarmellose Sodium", 4.0),
        ("Magnesium Stearate", 1.0),
        ("Povidone", 3.0),
        ("Colloidal Silicon Dioxide", 0.5),
        ("Water", 4.5),
    ],
    category="pharma",
)

disc = evaluate_discourse(tablet)
print(disc)

Three independent mechanisms flagged in one formulation:
1. **Lactose + Amlodipine**: Maillard reaction (amlodipine's primary amine + lactose reducing sugar)
2. **MgSt + Lactose**: Magnesium stearate catalyzes the Maillard reaction
3. **Povidone + Amlodipine**: Peroxide impurities in PVP oxidize amlodipine

A reformulator would replace lactose with mannitol and use a low-peroxide grade of PVP.

In [ ]:
# Chelation: ciprofloxacin + calcium (FDA-required labeling)
cipro = Formula(
    name="Ciprofloxacin with Antacid",
    ingredients=[
        ("Ciprofloxacin", 25.0), ("Calcium Carbonate", 30.0),
        ("Microcrystalline Cellulose", 30.0),
        ("Magnesium Stearate", 1.0), ("Water", 14.0),
    ],
    category="pharma",
)

disc = evaluate_discourse(cipro)
# Find the chelation violation
for topic in disc.topics:
    for claim in topic.claims:
        if claim.position == "violation":
            print(f"[VIOLATION] {topic.subject}")
            print(f"  {claim.detail}")
            print(f"  Evidence: {claim.evidence}")
            print(f"  Confidence: {claim.confidence}")

## 5. Ingredient Resolution

Under the hood, OpenMix resolves any INCI ingredient name to its molecular identity through a three-tier lookup: seed cache (2,400+ ingredients ship with the package), user cache, and PubChem API fallback.

In [ ]:
from openmix.resolver import resolve

for name in ["Niacinamide", "Retinol", "Sodium Lauryl Sulfate", "Cetrimonium Chloride"]:
    r = resolve(name)
    if r.resolved:
        print(f"{name}:")
        print(f"  SMILES: {r.smiles}")
        print(f"  LogP: {r.log_p}, MW: {r.molecular_weight}, Charge: {r.charge_type}")
        print()

## 6. Two Modes: Engineering vs Discovery

The same observation engine serves two goals. Same observations, different interpretation.

In [ ]:
obs_eng = observe(cream, mode="engineering")
obs_disc = observe(cream, mode="discovery")

print(f"Engineering mode: {obs_eng.concern_count:.1f} concerns (minimize to zero)")
print(f"Discovery mode:  {obs_disc.concern_count:.1f} concerns (only hard violations count)")
print(f"")
print(f"Soft violations become signals:  {len(obs_disc.signals)} signals to investigate")
print(f"Low-confidence discrepancies:    {len(obs_disc.discoveries)} knowledge gaps")

| | Engineering | Discovery |
|---|---|---|
| **Goal** | Zero concerns | Investigate surprises |
| **Hard violations** | Block (safety) | Block (safety) |
| **Soft violations** | Penalize | Surface as signals |
| **Discrepancies** | Fix them | Investigate them |

## 7. Knowledge Base

OpenMix knowledge lives in YAML. Chemists contribute domain expertise without writing code.

In [ ]:
from openmix import load_knowledge

kb = load_knowledge()
hard = len(kb.hard_rules)
soft = len(kb.soft_rules)

print(f"Interaction rules: {len(kb.interaction_rules)} ({hard} hard + {soft} soft)")
print(f"Oil HLB entries:   {len(kb.oil_hlb)}")
print()

# Rules by domain
categories = {}
for rule in kb.interaction_rules:
    categories[rule.category] = categories.get(rule.category, 0) + 1
for cat, count in sorted(categories.items(), key=lambda x: -x[1]):
    print(f"  {cat}: {count} rules")

## Next Steps

**Autonomous experiments** (requires an API key):
```python
from openmix import Experiment
result = Experiment.from_brief("Design a stable vitamin C serum").run()
```

**CLI**:
```bash
openmix discourse formula.yaml            # Multi-perspective evaluation
openmix observe formula.yaml              # Physics observations
openmix run "Design a stable serum"       # Autonomous experiment
openmix memory --discoveries              # View accumulated findings
```

**MCP server** (for AI agent integration):
```bash
python -m openmix.mcp_server
```

**Links**:
- [GitHub](https://github.com/vijayvkrishnan/openmix)
- [PyPI](https://pypi.org/project/openmix/)
- [Contributing Guide](https://github.com/vijayvkrishnan/openmix/blob/main/CONTRIBUTING.md) -- knowledge base contributions need YAML, not code

---
*OpenMix: The lab does the testing. OpenMix does the noticing.*